# Does the CRISM cleaning do the right thing

This notebook takes one CRISM multispectral survey observation, runs `src/preprocessing/crism/` over it, and looks at the result from every angle that would show a mistake. It answers nothing quantitatively: it is here so a person can see whether the cleaning is sane before a dataset is built on it.

One observation at a time. An image is about 37 MB and the cleaning takes a minute or so.

## Setup

In [ ]:
"""Import the picker and the panels that read what it cleaned."""

from visualization.preprocessing.panels import (
    checks,
    colour,
    labels,
    spectra,
    values,
    waves,
)
from visualization.preprocessing.picker import StripPicker

## Pick an observation

The list is drawn at random from the multispectral survey products the coverage stage already selected, so the check runs over the same data the dataset would be built from rather than over the archive at large.

Pressing Clean fetches the observation the first time, then loads it and runs every step. Every panel below refills itself whenever a new observation is confirmed, so none of the cells need running again.

In [ ]:
"""Offer a handful of observations and clean whichever is confirmed."""

picker = StripPicker()
picker.choose()

## 1. What the label says

Before looking at any number, check the observation is the kind this stage was written for. The mode has to be `MULTISPEC`, the sensor has to be the infrared one, and the unit has to be `I_OVER_F`, which is what makes values outside 0 to 1 impossible.

The three correction flags should all read `OFF`. They are why the atmosphere and the sun angle are still in the spectra, and why nothing further down should be surprised to find them there.

The last row is the one worth pausing on. On a hyperspectral product `MRO:HDF_SOFTWARE_NAME` names the filter the CRISM pipeline ran; on a multispectral one it reads `N/A`, because that filter was never run. The despiking in this stage is not repeating work someone else did.

In [ ]:
"""Tabulate the label fields worth checking before trusting anything else."""

picker.show_panel(labels.plot)

## 2. Where the bands sit

The band table is hardcoded, so the first thing to confirm is that it still describes the file. The left panel should rise from left to right without exception: the archive stores bands from long wavelength to short, and reading them in the wrong direction would put every spectrum backwards while leaving all the numbers looking plausible.

The crosses are bands the cleaning refuses to use, and there should be two of them: one inside the window where the atmosphere is opaque, one above the wavelength where the surface starts to glow rather than reflect.

The right panel shows the gaps. Multispectral survey mode downlinks selected channels rather than all of them, so the spacing is uneven by design, and the wide gaps mark where whole stretches of the spectrum were never sent down.

In [ ]:
"""Draw where the bands sit and how far apart they are."""

picker.show_panel(waves.plot)

## 3. The values, and what cannot be read

Top left counts what came off disk outside the range reflectance can take. The negative count matters most: the code this stage is ported from tests only for values above a thousand, so every one of those negatives would have passed straight through into the dataset.

The other three panels ask where the unreadable voxels are. Bottom left is the one to read carefully. The four columns drawn in red are the ones the wavelength table never calibrated, and they should stand out in the measured data too. If they do not, or if some other column does, the hardcoded list has stopped matching the archive.

In [ ]:
"""Draw the spread of the values and the shape of what is unreadable."""

picker.show_panel(values.plot)

## 4. The strip in false colour

The fastest check there is. If the band ordering were wrong, or the interleave misread, or the wavelengths off, this would not look like ground. It should read as terrain: a long thin strip with texture that follows landforms rather than the detector.

Comparing the two sides shows what the filtering did spatially. Vertical streaks running the length of the strip are detector columns rather than geology, and should be fainter on the right.

In [ ]:
"""Draw the strip in false colour before and after the filtering."""

picker.show_panel(colour.plot)

## 5. Spectra at every stage

This is where over-cleaning shows first, and it is the reason the stages are kept rather than overwritten.

Each line is the same pixel as one step of the cleaning left it. The steps should move single channels that stick out from their neighbours and leave everything else alone. What they must not do is round off the broad dips: those are absorption bands, they are what the whole dataset exists to capture, and a filter window that is too wide will quietly flatten them.

The fourth panel averages the whole strip, where the shape of the spectrum is clearer than in any single pixel.

In [ ]:
"""Draw a few spectra as every stage of the cleaning left them."""

picker.show_panel(spectra.plot)

## 6. What each step changed

A step that touches almost nothing is not doing its job. A step that touches a large share of the cube is doing somebody else's. Both are visible here, and the last column separates values that moved from values that stopped being readable at all, which are different failures.

As a rough expectation on ordinary data, the two filtering steps each move a couple of per cent of the cube, and neither should make anything unreadable that was readable before.

In [ ]:
"""Tabulate how much each step of the cleaning changed."""

picker.show_panel(checks.accounting)

## 7. Did the cleaning do the right amount

Four questions at once.

The two top panels draw the average spectrum of every detector column, before and after the destriping. A pushbroom instrument has one detector element per column per band, so a fault in one shows in every line at once and appears here as a column wandering away from the rest. The spread between them should be smaller on the right, and the printed number says by how much.

Bottom left is a physics check rather than a software one. Nothing in this stage corrects for the atmosphere, so the CO2 band near 2000 nm has to still be there. If it is missing, the band mapping is wrong somewhere upstream, whatever the other panels say.

Bottom right plots every kept value against itself. Almost all of it should sit on the diagonal. The points that leave it are exactly what the cleaning rewrote, and a correlation far below one means it rewrote too much.

In [ ]:
"""Draw the checks that catch a cleaning doing too much or too little."""

picker.show_panel(checks.plot)

## 8. Do other observations look the same

Everything above reads one observation, which is enough to see whether the cleaning works but not whether the assumptions behind it generalise. The band table and the list of dead columns are hardcoded, and they only hold while every multispectral survey observation is shaped the same way.

This cell checks that across several of them without downloading any images, since a label is a few kilobytes. It is the only reason to want more than one observation, and it does not cost one.

The wavelength file is expected to disagree: observations name different versions of it. That is harmless, and the last column says why.

In [ ]:
"""Compare the labels of several observations, without fetching their images."""

labels.across(count=6)